# Notebook 8: Physical Realizability (EOT)

Evaluate physical realizability of adversarial perturbations using the Maritime Environment Over Transformation (EOT) model.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from helpers import load_scenario, get_all_detections, MaritimeEOTDF

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 8)

## 8.1 Load Data

In [ ]:
SCENARIO = 'scenario2'
loader = load_scenario(SCENARIO)
detections = get_all_detections(loader)

ir_detections = detections[3]
print('IR Camera:', len(ir_detections), 'detections')

## 8.2 Initialize EOT Model

In [ ]:
eot = MaritimeEOTDF(
    wave_height=1.0,
    rain_rate=0.0,
    fog_visibility=10000.0
)
print('EOT model initialized')

## 8.3 Evaluate Realizability for Different Perturbations

In [ ]:
perturbations = np.linspace(-0.2, 0.2, 21)
realizability = []

sample_det = {
    'time': 0.0,
    'x_piren': 100.0,
    'y_piren': 50.0
}

for p in perturbations:
    score = eot.transform_detection(sample_det, p)
    realizability.append(score)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(perturbations, realizability, 'o-', linewidth=2, markersize=6)
ax.axhline(y=0.5, color='gray', linestyle='--', label='Threshold')
ax.set_xlabel('Perturbation (rad)')
ax.set_ylabel('Realizability Score')
ax.set_title('Physical Realizability vs Perturbation Magnitude')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

## 8.4 Realizability Under Different Weather Conditions

In [ ]:
conditions = [
    ('Calm', 0.5, 0.0, 10000),
    ('Moderate Waves', 1.5, 0.0, 10000),
    ('Heavy Rain', 1.0, 10.0, 5000),
    ('Fog', 1.0, 0.0, 500),
    ('Storm', 2.5, 20.0, 200)
]

perturbations = np.linspace(-0.1, 0.1, 11)

fig, ax = plt.subplots(figsize=(12, 6))

for name, wave, rain, fog in conditions:
    eot_cond = MaritimeEOTDF(wave_height=wave, rain_rate=rain, fog_visibility=fog)
    scores = [eot_cond.transform_detection(sample_det, p) for p in perturbations]
    ax.plot(perturbations, scores, 'o-', label=name, linewidth=2, markersize=5)

ax.set_xlabel('Perturbation (rad)')
ax.set_ylabel('Realizability Score')
ax.set_title('Realizability Under Different Maritime Conditions')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

## 8.5 Realizability Heatmap

In [ ]:
wave_heights = np.linspace(0, 3, 20)
perturbations = np.linspace(-0.2, 0.2, 20)

heatmap = np.zeros((len(wave_heights), len(perturbations)))

for i, wh in enumerate(wave_heights):
    eot_hm = MaritimeEOTDF(wave_height=wh, rain_rate=0, fog_visibility=10000)
    for j, p in enumerate(perturbations):
        heatmap[i, j] = eot_hm.transform_detection(sample_det, p)

fig, ax = plt.subplots(figsize=(12, 8))
im = ax.imshow(heatmap, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1,
               extent=[perturbations.min(), perturbations.max(), wave_heights.min(), wave_heights.max()],
               origin='lower')
ax.set_xlabel('Perturbation (rad)')
ax.set_ylabel('Wave Height (m)')
ax.set_title('Realizability Heatmap: Wave Height vs Perturbation')
plt.colorbar(im, ax=ax, label='Realizability Score')
plt.tight_layout()
plt.show()